In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 1: The GPU Has Two Masters — VRAM and Bandwidth

*Part of the Vizuara AI/ML series on LLM Inference Internals*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Every time you use ChatGPT, Claude, or any large language model, there is a GPU somewhere doing the actual work. And that GPU is almost certainly not running at full speed.

This is not a bug. It is a fundamental consequence of how GPUs are built — and understanding it is the key to understanding why LLM inference is slow, why KV caching helps (and also hurts), and why engineers at AI labs spend enormous effort on things that look, at first glance, like minor implementation details.

In this notebook, we will build up a precise, quantitative picture of the GPU's two fundamental constraints:

1. **Arithmetic throughput** — how many operations per second the chip can perform
2. **Memory bandwidth** — how quickly data can travel from VRAM into the chip

We will then learn the **roofline model** — a simple but powerful framework that tells you, for any computation, which of these two constraints is the binding one. By the end, you will be able to look at any deep learning operation and immediately classify it as either *memory-bound* or *compute-bound*. That classification is the foundation of everything that follows in this series.

Here is a teaser of what we will produce:

In [ ]:
# A preview of our final output — a beautiful roofline diagram
# with real GPU specs and real LLM operations plotted on it.
# We will build every piece of this from scratch.
import matplotlib.pyplot as plt
import numpy as np
print("All imports ready. Let us build something beautiful.")

## 2. Building Intuition

Before we write a single line of code or look at a single equation, let us make sure the *idea* is crystal clear. The best engineers we know say that once you truly internalize this section, the math becomes almost obvious.

### The Shipyard Analogy

Imagine you run a shipyard. Your shipyard is extraordinarily capable — it can assemble **1,000 units per second** when running at full capacity. You are very proud of it.

Now, your raw materials come from a warehouse across town. Delivery trucks shuttle back and forth between the warehouse and your shipyard. Each truck can carry **100 units**, and the trucks arrive at a rate of **1 per second**.

**Question:** How many units per second does your shipyard actually produce?

Not 1,000. It produces **100 units per second** — exactly as fast as the trucks can deliver materials. The shipyard sits idle for 90% of the time, waiting. It does not matter how fast the assembly line is. The truck is the bottleneck.

This is the **memory-bound regime**.

Now imagine the opposite: you somehow arrange for trucks to arrive 20 times per second, delivering 2,000 units per second. Now the shipyard cannot keep up. The limiting factor is the assembly line's speed of 1,000 units per second.

This is the **compute-bound regime**.

### Mapping the Analogy to a GPU

| Shipyard Concept | GPU Concept |
|---|---|
| Assembly line | Arithmetic units (CUDA cores, tensor cores) |
| Assembly speed (units/sec) | **Arithmetic throughput** (FLOP/s) |
| Warehouse | **VRAM** (GPU memory, e.g. 40 GB on an A100) |
| Truck delivery speed (units/sec) | **Memory bandwidth** (bytes/sec, e.g. 2 TB/s) |
| Raw materials | Model weights, activations, KV cache entries |

The GPU's silicon — the part that does multiplications and additions — is the assembly line. VRAM is the warehouse. The data bus connecting them is the truck.

### The Key Insight

Every single computation you run on a GPU has an inherent ratio: **how many arithmetic operations does it perform per byte of data it loads from memory?** This ratio is called **arithmetic intensity**, and it completely determines which of the two constraints bites you.

- **Low arithmetic intensity** → you are moving lots of data but doing little math with each byte → truck is the bottleneck → **memory-bound**
- **High arithmetic intensity** → you are doing lots of math per byte of data → shipyard is the bottleneck → **compute-bound**

### 🤔 Think About This

Before reading on, pause and consider: a matrix multiplication where you multiply a (1 × 4096) vector by a (4096 × 4096) weight matrix. You load roughly 4096 × 4096 × 2 bytes ≈ 32 MB of weights. You perform roughly 2 × 4096 × 4096 ≈ 33 million arithmetic operations.

That is about **1 operation per byte**. Does that feel like a lot of math relative to the data you are moving? We will calculate this precisely in the next section — and the answer will surprise you.

## 3. The Mathematics

Now let us make this precise. The roofline model is built on three numbers.

### 3.1 The Two Hardware Constants

For a given GPU, we have:

$$\pi = \text{Peak arithmetic throughput} \quad [\text{FLOP/s}]$$

$$\beta = \text{Peak memory bandwidth} \quad [\text{bytes/s}]$$

These are fixed properties of the chip. You cannot change them (short of buying different hardware).

For reference, here are real numbers for commonly used GPUs:

| GPU | Peak FP16 FLOP/s ($\pi$) | Memory Bandwidth ($\beta$) |
|---|---|---|
| NVIDIA T4 (Colab free tier) | 65 TFLOP/s | 300 GB/s |
| NVIDIA A100 80GB | 312 TFLOP/s | 2,000 GB/s |
| NVIDIA H100 SXM | 989 TFLOP/s | 3,350 GB/s |

### 3.2 The Arithmetic Intensity of a Computation

For any computation, we define its **arithmetic intensity** $I$ as:

$$I = \frac{W}{Q} \quad \left[\frac{\text{FLOP}}{\text{byte}}\right]$$

where:
- $W$ = total number of arithmetic operations (floating-point ops, or FLOPs)
- $Q$ = total bytes of data moved between VRAM and the chip

This equation says: *for every byte we drag across the memory bus, how many arithmetic operations do we perform on it?* A high $I$ means we are doing lots of math per byte (efficient use of the memory bus). A low $I$ means we are burning bandwidth to do very little math.

Computationally, this means: count your multiplications and additions, count your memory accesses, and divide.

### 3.3 The Roofline Bound

Given $I$, $\pi$, and $\beta$, the maximum achievable performance of any computation is:

$$P = \min\left(\pi, \; \beta \cdot I\right) \quad [\text{FLOP/s}]$$

This equation says: *your performance is limited by whichever of two things runs out first — the arithmetic units (capped at $\pi$), or the memory bus (which delivers $\beta$ bytes/s, each carrying $I$ FLOPs worth of work, so delivering $\beta \cdot I$ FLOP/s of useful compute).*

Computationally, this means: if $\beta \cdot I < \pi$, the memory bus is the bottleneck and you are memory-bound. If $\beta \cdot I \geq \pi$, the arithmetic units are the bottleneck and you are compute-bound.

### 3.4 The Ridge Point

The transition between these two regimes happens at the **ridge point** $I^*$:

$$I^* = \frac{\pi}{\beta} \quad \left[\frac{\text{FLOP}}{\text{byte}}\right]$$

This equation says: *the ridge point is where arithmetic demand exactly matches what the memory bus can supply.* If your computation's $I$ is above $I^*$, you are compute-bound. Below $I^*$, you are memory-bound.

For an A100: $I^* = \frac{312 \times 10^{12}}{2 \times 10^{12}} = 156$ FLOP/byte.

That means to keep an A100's arithmetic units fully fed, every byte you load must be used for 156 arithmetic operations. That is a very high bar.

## 4. Let's Build It — Component by Component

### 4.1 GPU Specification Dataclass

Let us start by encoding real GPU specifications as Python objects. This makes everything concrete and queryable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style for beautiful plots
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#e6edf3',
    'ytick.color': '#e6edf3',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.linewidth': 0.8,
    'font.family': 'monospace',
    'axes.titlecolor': '#e6edf3',
})

np.random.seed(42)

@dataclass
class GPUSpec:
    """
    Encodes the two fundamental hardware constants that govern
    every computation on a GPU.
    """
    name: str
    peak_tflops: float        # Arithmetic throughput in TFLOP/s (FP16)
    bandwidth_tbs: float      # Memory bandwidth in TB/s
    vram_gb: float            # Total VRAM in GB

    @property
    def peak_flops(self) -> float:
        """Peak throughput in raw FLOP/s"""
        return self.peak_tflops * 1e12

    @property
    def bandwidth_bytes_per_sec(self) -> float:
        """Bandwidth in bytes/s"""
        return self.bandwidth_tbs * 1e12

    @property
    def ridge_point(self) -> float:
        """
        The arithmetic intensity (FLOP/byte) at the transition
        between memory-bound and compute-bound regimes.
        I* = π / β
        """
        return self.peak_flops / self.bandwidth_bytes_per_sec

    def achieved_performance(self, arithmetic_intensity: float) -> float:
        """
        Given arithmetic intensity I (FLOP/byte), return the
        maximum achievable performance in FLOP/s.
        P = min(π, β · I)
        """
        memory_limited = self.bandwidth_bytes_per_sec * arithmetic_intensity
        return min(self.peak_flops, memory_limited)

    def utilization(self, arithmetic_intensity: float) -> float:
        """Fraction of peak compute actually used (0 to 1)"""
        return self.achieved_performance(arithmetic_intensity) / self.peak_flops

    def __str__(self):
        return (f"{self.name}: {self.peak_tflops} TFLOP/s | "
                f"{self.bandwidth_tbs} TB/s | "
                f"Ridge={self.ridge_point:.1f} FLOP/byte | "
                f"{self.vram_gb}GB VRAM")


# Real GPU specifications — every number here is from official datasheets
gpus = {
    "T4":  GPUSpec("NVIDIA T4",       peak_tflops=65,   bandwidth_tbs=0.300, vram_gb=16),
    "A100": GPUSpec("NVIDIA A100 80GB", peak_tflops=312,  bandwidth_tbs=2.000, vram_gb=80),
    "H100": GPUSpec("NVIDIA H100 SXM", peak_tflops=989,  bandwidth_tbs=3.350, vram_gb=80),
}

for name, gpu in gpus.items():
    print(gpu)

Let us pause and think about what the ridge point numbers tell us. The T4 has a ridge point of about 217 FLOP/byte, the A100 is 156 FLOP/byte, and the H100 is about 295 FLOP/byte.

These numbers mean: *to fully utilize a T4's arithmetic units, every byte you load from VRAM must be used for 217 operations.* As we will soon see, many LLM inference operations fall far short of this.

### 4.2 Computing Arithmetic Intensity for Real Operations

Now let us compute $I$ for the operations that actually happen inside a transformer during inference. We will focus on the two most important ones: **matrix-vector multiplication** (the dominant operation during token generation) and **matrix-matrix multiplication** (the dominant operation during the prefill phase).

In [ ]:
@dataclass
class ComputeOperation:
    """
    Represents a single GPU operation with its compute and memory costs.
    We compute arithmetic intensity from first principles.
    """
    name: str
    description: str
    flops: float         # Total floating-point operations
    bytes_moved: float   # Total bytes transferred from VRAM

    @property
    def arithmetic_intensity(self) -> float:
        """I = W / Q  (FLOP per byte)"""
        return self.flops / self.bytes_moved

    def __str__(self):
        return (f"[{self.name}]\n"
                f"  FLOPs:     {self.flops:.3e}\n"
                f"  Bytes:     {self.bytes_moved:.3e}\n"
                f"  Intensity: {self.arithmetic_intensity:.2f} FLOP/byte")


def matvec_operation(M: int, N: int, dtype_bytes: int = 2) -> ComputeOperation:
    """
    Matrix-vector multiplication: y = W x
    W is shape (M, N), x is shape (N,), y is shape (M,)

    FLOPs: 2 * M * N
      - One multiply and one add per element of W
      - Factor of 2 because each W[i,j] contributes one MAC (multiply-accumulate)

    Bytes moved: (M * N + N + M) * dtype_bytes
      - Load W: M * N elements
      - Load x: N elements
      - Store y: M elements
    """
    flops = 2 * M * N
    bytes_moved = (M * N + N + M) * dtype_bytes
    return ComputeOperation(
        name=f"MatVec ({M}x{N})",
        description=f"y=Wx, W:{M}x{N}, x:{N}x1",
        flops=flops,
        bytes_moved=bytes_moved
    )


def matmat_operation(M: int, N: int, K: int, dtype_bytes: int = 2) -> ComputeOperation:
    """
    Matrix-matrix multiplication: C = A @ B
    A is shape (M, K), B is shape (K, N), C is shape (M, N)

    FLOPs: 2 * M * N * K
      - For each of M*N output elements, we do K multiply-accumulates

    Bytes moved: (M*K + K*N + M*N) * dtype_bytes
      - Load A, load B, store C
    """
    flops = 2 * M * N * K
    bytes_moved = (M * K + K * N + M * N) * dtype_bytes
    return ComputeOperation(
        name=f"MatMat ({M}x{K}) @ ({K}x{N})",
        description=f"C=AB, A:{M}x{K}, B:{K}x{N}",
        flops=flops,
        bytes_moved=bytes_moved
    )


# A typical LLM linear layer: hidden_dim = 4096
D = 4096

# During generation: one token at a time (batch size = 1)
# The query/key/value projections look like this
gen_op = matvec_operation(M=D, N=D)
print("=== Token Generation (batch=1) ===")
print(gen_op)
print()

# During prefill: processing a full 2000-token prompt at once
SEQ_LEN = 2000
prefill_op = matmat_operation(M=SEQ_LEN, N=D, K=D)
print(f"=== Prompt Prefill (seq_len={SEQ_LEN}) ===")
print(prefill_op)

Now let us visualize how these operations compare to each GPU's ridge point. This is where it gets very interesting.

### 4.3 Visualizing Operations on the Roofline

In [ ]:
def plot_roofline(gpu: GPUSpec, operations: List[ComputeOperation],
                  ax=None, title_suffix=""):
    """
    Draws the roofline diagram for a given GPU and plots
    operations as points on it.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    # Build the roofline curve
    # x-axis: arithmetic intensity (FLOP/byte), log scale
    x = np.logspace(-2, 4, 1000)
    y = np.minimum(gpu.peak_flops * np.ones_like(x),
                   gpu.bandwidth_bytes_per_sec * x)

    # Convert to TFLOP/s for readability
    y_tflops = y / 1e12

    # Plot the roofline
    ax.loglog(x, y_tflops, color='#58a6ff', linewidth=2.5, zorder=3)

    # Shade the memory-bound region
    memory_bound_mask = x <= gpu.ridge_point
    ax.fill_between(x[memory_bound_mask],
                    y_tflops[memory_bound_mask],
                    alpha=0.08, color='#f85149', zorder=1,
                    label='Memory-bound region')

    # Shade the compute-bound region
    compute_bound_mask = x >= gpu.ridge_point
    ax.fill_between(x[compute_bound_mask],
                    y_tflops[compute_bound_mask],
                    alpha=0.08, color='#3fb950', zorder=1,
                    label='Compute-bound region')

    # Mark the ridge point
    ax.axvline(x=gpu.ridge_point, color='#e3b341', linewidth=1.5,
               linestyle='--', alpha=0.7, zorder=2)
    ax.text(gpu.ridge_point * 1.1, gpu.peak_tflops * 0.85,
            f'Ridge\nI*={gpu.ridge_point:.0f}',
            color='#e3b341', fontsize=8, va='top')

    # Label the flat ceiling
    ax.axhline(y=gpu.peak_tflops, color='#3fb950', linewidth=1,
               linestyle=':', alpha=0.5, zorder=2)
    ax.text(x[-1] * 0.6, gpu.peak_tflops * 1.05,
            f'Peak: {gpu.peak_tflops} TFLOP/s',
            color='#3fb950', fontsize=8, ha='right')

    # Label the bandwidth slope
    slope_x = gpu.ridge_point * 0.05
    slope_y = gpu.bandwidth_bytes_per_sec * slope_x / 1e12
    ax.text(slope_x, slope_y * 2,
            f'Slope = β\n{gpu.bandwidth_tbs} TB/s',
            color='#f85149', fontsize=8, rotation=35, va='bottom')

    # Plot each operation as a colored dot
    colors = ['#ff7b72', '#ffa657', '#d2a8ff', '#79c0ff', '#56d364']
    markers = ['o', 's', '^', 'D', 'v']

    for i, op in enumerate(operations):
        I = op.arithmetic_intensity
        perf = gpu.achieved_performance(I) / 1e12  # TFLOP/s
        util = gpu.utilization(I)

        color = colors[i % len(colors)]
        ax.scatter(I, perf, color=color, s=120, zorder=5,
                   marker=markers[i % len(markers)],
                   edgecolors='white', linewidth=1.0)
        ax.annotate(f'{op.name}\n(I={I:.1f}, util={util:.1%})',
                    xy=(I, perf),
                    xytext=(I * 0.3, perf * 0.3),
                    fontsize=7.5,
                    color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=1),
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22',
                              edgecolor=color, alpha=0.9))

    ax.set_xlabel('Arithmetic Intensity  (FLOP / byte)', fontsize=11)
    ax.set_ylabel('Achieved Performance  (TFLOP/s)', fontsize=11)
    ax.set_title(f'Roofline Model — {gpu.name}{title_suffix}', fontsize=13, pad=15)
    ax.grid(True, which='both', alpha=0.3)

    mem_patch = mpatches.Patch(color='#f85149', alpha=0.3, label='Memory-Bound')
    cmp_patch = mpatches.Patch(color='#3fb950', alpha=0.3, label='Compute-Bound')
    ax.legend(handles=[mem_patch, cmp_patch], loc='lower right', fontsize=9)

    return ax


# Let us plot the A100 roofline with our two operations
fig, ax = plt.subplots(figsize=(11, 7))

ops_to_plot = [
    matvec_operation(M=4096, N=4096),            # Token generation
    matmat_operation(M=2000, N=4096, K=4096),    # Prefill
]

plot_roofline(gpus["A100"], ops_to_plot, ax=ax,
              title_suffix=" — Token Generation vs Prefill")

plt.tight_layout()
plt.savefig('roofline_a100.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("\n📊 Roofline diagram saved!")

Look at where those two points land. The matrix-vector operation (token generation) has an arithmetic intensity of about **1 FLOP/byte** — it sits far, far to the left in the memory-bound regime. The A100's ridge point is at 156 FLOP/byte, and we are hitting 1. That means we are achieving roughly **1/156 ≈ 0.6% of the GPU's peak arithmetic throughput** during token generation.

The prefill operation fares much better because we process 2,000 tokens simultaneously — but even it may not reach the ridge point depending on the exact dimensions.

This is the core tension of LLM inference. Let us now build a tool to explore how arithmetic intensity varies with key parameters.

## 5. 🔧 Your Turn

### TODO: Implement `compute_attention_intensity`

The attention mechanism has its own arithmetic intensity — and it changes with sequence length in an interesting way. Your task is to implement a function that computes the arithmetic intensity of the **attention score computation** for a single attention head.

Recall: attention computes scores as $\text{scores} = QK^\top / \sqrt{d_k}$, where $Q$ and $K$ are both shape `(seq_len, d_k)`. The result is shape `(seq_len, seq_len)`.

In [ ]:
def compute_attention_intensity(seq_len: int, d_k: int,
                                 dtype_bytes: int = 2) -> float:
    """
    Compute the arithmetic intensity of QK^T for one attention head.

    Args:
        seq_len: Number of tokens (both queries and keys)
        d_k:     Head dimension (typically hidden_dim / num_heads)
        dtype_bytes: Bytes per element (2 for FP16, 4 for FP32)

    Returns:
        Arithmetic intensity in FLOP/byte

    Hints:
    ============ TODO ============
    Step 1: Count the FLOPs for QK^T
            - Q is (seq_len, d_k), K^T is (d_k, seq_len)
            - This is a matrix multiplication of shape (seq_len, d_k) @ (d_k, seq_len)
            - FLOPs for a (M, K) @ (K, N) matmul = 2 * M * K * N
            - Here M = seq_len, K = d_k, N = seq_len
            - So FLOPs = 2 * seq_len * d_k * seq_len

    Step 2: Count the bytes moved
            - We must LOAD Q: seq_len * d_k elements
            - We must LOAD K: seq_len * d_k elements
            - We must STORE scores: seq_len * seq_len elements
            - Total bytes = (2 * seq_len * d_k + seq_len * seq_len) * dtype_bytes

    Step 3: Return FLOPs / bytes
    ============ END TODO ========
    """
    # ============ TODO ============
    flops = ???  # YOUR CODE HERE
    bytes_moved = ???  # YOUR CODE HERE
    # ============ END TODO ========
    return flops / bytes_moved


# ✅ Verification cell
# Run this after implementing the function above
def verify_attention_intensity():
    # For seq_len=1, d_k=64: FLOPs = 2*1*64*1 = 128
    # bytes = (2*1*64 + 1*1) * 2 = (128 + 1)*2 = 258
    # intensity = 128/258 ≈ 0.496
    result = compute_attention_intensity(seq_len=1, d_k=64)
    expected = (2 * 1 * 64 * 1) / ((2 * 1 * 64 + 1 * 1) * 2)
    assert abs(result - expected) < 1e-6, f"Got {result}, expected {expected}"

    # For large seq_len, intensity should grow (approaches d_k/2)
    small = compute_attention_intensity(seq_len=10, d_k=64)
    large = compute_attention_intensity(seq_len=1000, d_k=64)
    assert large > small, "Intensity should increase with seq_len for large seq_len"

    print("✅ All checks passed! Your implementation is correct.")
    print(f"   Intensity at seq_len=10:   {small:.3f} FLOP/byte")
    print(f"   Intensity at seq_len=1000: {large:.3f} FLOP/byte")
    print(f"   (Notice how it grows — larger batches help!)")

verify_attention_intensity()

### TODO: Implement `gpu_utilization_report`

Now let us build a function that, given a list of operations and a GPU, prints a clear utilization report. This is the kind of tool a real ML engineer would write to profile inference.

In [ ]:
def gpu_utilization_report(gpu: GPUSpec,
                            operations: List[ComputeOperation]) -> None:
    """
    Print a formatted report showing how efficiently each operation
    uses the given GPU, and whether it is memory-bound or compute-bound.

    Args:
        gpu:        A GPUSpec object
        operations: List of ComputeOperation objects

    For each operation, print:
      - The operation name
      - Its arithmetic intensity
      - Whether it is memory-bound or compute-bound
      - Its utilization as a percentage of peak compute
      - A simple ASCII bar chart of utilization

    ============ TODO ============
    Step 1: Print a header line with the GPU name

    Step 2: For each operation:
        a. Get its arithmetic_intensity property
        b. Call gpu.utilization(intensity) to get utilization (0 to 1)
        c. Compare intensity to gpu.ridge_point to determine the regime
        d. Create a bar: '█' * int(utilization * 20) gives a 20-char bar
        e. Print all of this in a readable format

    Step 3: Print a footer note explaining the ridge point
    ============ END TODO ========
    """
    # ============ TODO ============
    # YOUR CODE HERE
    pass
    # ============ END TODO ========


# ✅ Verification
# After implementing, running this should print a formatted table
sample_ops = [
    matvec_operation(M=4096, N=4096),
    matmat_operation(M=32,   N=4096, K=4096),
    matmat_operation(M=512,  N=4096, K=4096),
    matmat_operation(M=2048, N=4096, K=4096),
]
gpu_utilization_report(gpus["A100"], sample_ops)

## 6. Putting It All Together

Now we will combine everything into a comprehensive analysis: how does the arithmetic intensity of LLM operations vary with batch size and sequence length? This is the central question for understanding inference efficiency.

In [ ]:
def analyze_inference_regimes(gpu: GPUSpec, d_model: int = 4096) -> None:
    """
    Sweeps across batch sizes and shows how arithmetic intensity
    changes — and where the crossover from memory-bound to
    compute-bound happens.
    """
    batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]

    intensities = []
    for bs in batch_sizes:
        op = matmat_operation(M=bs, N=d_model, K=d_model)
        intensities.append(op.arithmetic_intensity)

    intensities = np.array(intensities)
    utilizations = np.array([gpu.utilization(I) / gpu.peak_flops * gpu.peak_flops
                              for I in intensities])
    util_fractions = intensities / gpu.ridge_point
    util_fractions = np.minimum(util_fractions, 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'LLM Inference Efficiency vs Batch Size\n{gpu.name} | d_model={d_model}',
                 fontsize=13, y=1.02)

    # Left plot: Arithmetic intensity vs batch size
    ax1 = axes[0]
    ax1.semilogx(batch_sizes, intensities, 'o-', color='#58a6ff',
                 linewidth=2, markersize=7, markerfacecolor='white',
                 markeredgewidth=2, zorder=3)
    ax1.axhline(y=gpu.ridge_point, color='#e3b341', linewidth=2,
                linestyle='--', label=f'Ridge point (I*={gpu.ridge_point:.0f})', zorder=2)

    # Color the regions
    ax1.fill_between(batch_sizes,
                     [min(I, gpu.ridge_point) for I in intensities],
                     alpha=0.15, color='#f85149', label='Memory-bound portion')

    # Annotate the crossover
    crossover_bs = None
    for i, (bs, I) in enumerate(zip(batch_sizes, intensities)):
        if I >= gpu.ridge_point:
            crossover_bs = bs
            ax1.axvline(x=bs, color='#3fb950', linewidth=1.5,
                        linestyle=':', alpha=0.7)
            ax1.text(bs * 1.1, gpu.ridge_point * 0.7,
                     f'Crossover\nat bs={bs}',
                     color='#3fb950', fontsize=9)
            break

    ax1.set_xlabel('Batch Size', fontsize=11)
    ax1.set_ylabel('Arithmetic Intensity  (FLOP/byte)', fontsize=11)
    ax1.set_title('Arithmetic Intensity vs Batch Size', fontsize=11)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(batch_sizes)
    ax1.set_xticklabels([str(b) for b in batch_sizes], rotation=45, fontsize=8)

    # Right plot: GPU utilization fraction
    ax2 = axes[1]
    bar_colors = ['#f85149' if u < 0.5 else '#e3b341' if u < 0.9 else '#3fb950'
                  for u in util_fractions]
    bars = ax2.bar(range(len(batch_sizes)), util_fractions * 100,
                   color=bar_colors, edgecolor='#30363d', linewidth=0.5, zorder=3)

    ax2.set_xlabel('Batch Size', fontsize=11)
    ax2.set_ylabel('GPU Utilization  (% of peak compute)', fontsize=11)
    ax2.set_title('Compute Utilization vs Batch Size', fontsize=11)
    ax2.set_xticks(range(len(batch_sizes)))
    ax2.set_xticklabels([str(b) for b in batch_sizes], rotation=45, fontsize=8)
    ax2.axhline(y=100, color='#3fb950', linewidth=1.5, linestyle='--',
                alpha=0.7, label='Peak (100%)')
    ax2.set_ylim(0, 110)
    ax2.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for i, (bar, util) in enumerate(zip(bars, util_fractions)):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1.5,
                 f'{util * 100:.1f}%',
                 ha='center', va='bottom', fontsize=6.5,
                 color='#e6edf3')

    # Legend for colors
    red_patch = mpatches.Patch(color='#f85149', label='< 50% util (severely memory-bound)')
    yellow_patch = mpatches.Patch(color='#e3b341', label='50–90% util')
    green_patch = mpatches.Patch(color='#3fb950', label='> 90% util (near compute-bound)')
    ax2.legend(handles=[red_patch, yellow_patch, green_patch], fontsize=8, loc='upper left')

    plt.tight_layout()
    plt.savefig('inference_efficiency.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    print(f"\n📊 Analysis complete!")
    if crossover_bs:
        print(f"   The A100 becomes compute-bound at batch_size ≈ {crossover_bs}")
    else:
        print(f"   The {gpu.name} stays memory-bound across all tested batch sizes!")


analyze_inference_regimes(gpus["A100"])

## 7. Deep Dive — Comparing Across GPUs

Let us now run a systematic comparison across all three GPUs. This reveals something important: more powerful GPUs are *not automatically more efficient* for small-batch inference — they can actually have *higher* ridge points, which makes it harder to escape the memory-bound regime.

In [ ]:
def compare_gpus_roofline(gpu_dict: Dict[str, GPUSpec],
                           operations: List[ComputeOperation]) -> None:
    """
    Side-by-side roofline comparison across multiple GPUs.
    """
    n_gpus = len(gpu_dict)
    fig, axes = plt.subplots(1, n_gpus, figsize=(6 * n_gpus, 6), sharey=False)

    for ax, (name, gpu) in zip(axes, gpu_dict.items()):
        plot_roofline(gpu, operations, ax=ax)

    fig.suptitle('Roofline Model — GPU Comparison\nSame operations, different hardware',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('gpu_comparison_roofline.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()


# Operations representing key LLM inference scenarios
inference_ops = [
    matvec_operation(M=4096, N=4096),                   # Single-token generation
    matmat_operation(M=8,    N=4096, K=4096),            # Small batch generation
    matmat_operation(M=128,  N=4096, K=4096),            # Medium batch
    matmat_operation(M=2048, N=4096, K=4096),            # Full prefill
]

compare_gpus_roofline(gpus, inference_ops)

# Print a summary table
print("\n" + "="*65)
print(f"{'Operation':<30} {'T4 Util':>10} {'A100 Util':>10} {'H100 Util':>10}")
print("="*65)
for op in inference_ops:
    I = op.arithmetic_intensity
    utils = {name: f"{gpu.utilization(I):.1%}"
             for name, gpu in gpus.items()}
    print(f"{op.name:<30} {utils['T4']:>10} {utils['A100']:>10} {utils['H100']:>10}")
print("="*65)
print("Low utilization = memory-bound = waiting for data, not doing math")

The table above makes the core lesson viscerally clear. A single-token generation step (the matrix-vector multiply) achieves roughly 0.5% utilization on an A100 and even less on an H100. We are wasting the vast majority of the most expensive silicon on the planet — not because of bad code, but because of the fundamental arithmetic intensity of the computation.

This is *exactly* why the KV cache matters, and also why it creates new problems. But we are getting ahead of ourselves. Let us finish this section with our most satisfying output.

## 8. 🎯 Final Output — The Complete Roofline Dashboard

In [ ]:
def create_roofline_dashboard(gpu: GPUSpec) -> None:
    """
    A beautiful, comprehensive roofline dashboard showing:
    1. The roofline curve with all key operations
    2. How intensity scales with batch size (the "path to efficiency")
    3. Key numerical callouts
    """
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0d1117')

    # --- Main roofline (left, large) ---
    ax_main = fig.add_subplot(2, 2, (1, 3))

    # Build operations spanning the entire intensity spectrum
    all_ops = [
        # Name, M, N, K (K=None means matvec)
        ("Token gen (bs=1)",     1,    4096, None),
        ("Batch gen (bs=4)",     4,    4096, 4096),
        ("Batch gen (bs=32)",    32,   4096, 4096),
        ("Batch gen (bs=128)",   128,  4096, 4096),
        ("Prefill (s=512)",      512,  4096, 4096),
        ("Prefill (s=2048)",     2048, 4096, 4096),
    ]

    op_objects = []
    for label, M, N, K in all_ops:
        if K is None:
            op = matvec_operation(M=N, N=N)
            op.name = label
        else:
            op = matmat_operation(M=M, N=N, K=K)
            op.name = label
        op_objects.append(op)

    plot_roofline(gpu, op_objects, ax=ax_main,
                  title_suffix=" — LLM Inference Operations")

    # Annotate the "path to efficiency" arrow
    ax_main.annotate('', xy=(200, gpu.peak_tflops * 0.8),
                     xytext=(1, gpu.peak_tflops * 0.1),
                     arrowprops=dict(arrowstyle='->', color='#e6edf3',
                                     lw=2, connectionstyle='arc3,rad=-0.2'))
    ax_main.text(15, gpu.peak_tflops * 0.3, 'Path to\nEfficiency\n(↑ batch size)',
                 color='#e6edf3', fontsize=9, ha='center',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='#21262d',
                           edgecolor='#58a6ff', alpha=0.9))

    # --- Top right: Intensity vs batch size ---
    ax_batch = fig.add_subplot(2, 2, 2)
    batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048])
    intensities = np.array([matmat_operation(M=int(b), N=4096, K=4096).arithmetic_intensity
                             for b in batch_sizes])

    colors_line = ['#f85149' if I < gpu.ridge_point else '#3fb950' for I in intensities]
    for i in range(len(batch_sizes) - 1):
        ax_batch.semilogx(batch_sizes[i:i+2], intensities[i:i+2],
                          'o-', color=colors_line[i], linewidth=2,
                          markersize=5, markerfacecolor='white',
                          markeredgewidth=1.5)

    ax_batch.axhline(y=gpu.ridge_point, color='#e3b341', linewidth=2,
                     linestyle='--', label=f'Ridge I*={gpu.ridge_point:.0f}')
    ax_batch.set_xlabel('Batch Size', fontsize=10)
    ax_batch.set_ylabel('Arithmetic Intensity (FLOP/byte)', fontsize=10)
    ax_batch.set_title('Intensity vs Batch Size\n(Linear layer, d=4096)', fontsize=10)
    ax_batch.legend(fontsize=8)
    ax_batch.grid(True, alpha=0.3)
    ax_batch.set_xticks(batch_sizes)
    ax_batch.set_xticklabels([str(b) for b in batch_sizes], rotation=60, fontsize=7)

    # --- Bottom right: Key stats panel ---
    ax_stats = fig.add_subplot(2, 2, 4)
    ax_stats.axis('off')

    stats_text = f"""
  ╔══════════════════════════════════════╗
  ║  {gpu.name:<36}║
  ╠══════════════════════════════════════╣
  ║  Peak Compute:  {gpu.peak_tflops:>8.0f} TFLOP/s       ║
  ║  Bandwidth:     {gpu.bandwidth_tbs:>8.3f} TB/s          ║
  ║  VRAM:          {gpu.vram_gb:>8.0f} GB             ║
  ║  Ridge Point:   {gpu.ridge_point:>8.1f} FLOP/byte    ║
  ╠══════════════════════════════════════╣
  ║  Token gen utilization:   {gpu.utilization(matvec_operation(4096,4096).arithmetic_intensity):>6.2%}  ║
  ║  Prefill (s=2048) util:   {gpu.utilization(matmat_operation(2048,4096,4096).arithmetic_intensity):>6.2%}  ║
  ╠══════════════════════════════════════╣
  ║  ⚠  Token generation is deeply      ║
  ║     memory-bound. This is WHY       ║
  ║     KV caching matters — and why    ║
  ║     it creates new pressures.       ║
  ╚══════════════════════════════════════╝
    """

    ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes,
                  fontsize=8.5, verticalalignment='top', fontfamily='monospace',
                  color='#e6edf3',
                  bbox=dict(boxstyle='round,pad=0.5', facecolor='#161b22',
                            edgecolor='#30363d'))
    ax_stats.set_title('Hardware Summary', fontsize=10, pad=5)

    plt.suptitle(
        '🚀 The GPU Roofline Model — Understanding LLM Inference Bottlenecks\n'
        'Vizuara AI/ML Series | Section 1: The GPU Has Two Masters',
        fontsize=12, y=1.01, color='#e6edf3'
    )

    plt.tight_layout()
    plt.savefig('roofline_dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    print("\n🎉 Dashboard saved as 'roofline_dashboard.png'")
    print("   Screenshot this — it tells the whole story of Section 1!")


create_roofline_dashboard(gpus["A100"])

## 9. Reflection and Next Steps

### What We Built

In this notebook, we went from first principles all the way to a complete, quantitative picture of GPU compute constraints. Here is a summary of what we now know:

**The two fundamental constraints:**
- **Arithmetic throughput** ($\pi$): how many FLOP/s the chip can perform
- **Memory bandwidth** ($\beta$): how many bytes/s can travel from VRAM to the chip

**The roofline model:**
$$P = \min\left(\pi, \; \beta \cdot I\right)$$
where $I = W/Q$ is the arithmetic intensity of your computation.

**The central finding:** Token generation during LLM inference has an arithmetic intensity of roughly 1 FLOP/byte, while modern GPUs have ridge points of 150–300 FLOP/byte. We are operating at roughly **0.3–0.7% of peak compute**. The GPU is sitting idle, waiting for data.

### 🤔 Reflection Questions

Take a moment before moving on. Can you answer these from memory?

1. **What is arithmetic intensity?** Define it in one sentence using the words "operations" and "bytes".

2. **Why does batch size matter so much?** A batch-size-1 matrix-vector multiply has intensity ~1 FLOP/byte. What happens to intensity as you increase batch size toward the ridge point? *Hint: think about what changes in the numerator vs. denominator of $I = W/Q$.*

3. **Is a more powerful GPU always better for inference?** The H100 has a higher ridge point than the A100. What does that mean for single-token generation — is it easier or harder to reach good utilization?

4. **What does the KV cache change about this picture?** We established that reading model weights is the bottleneck during token generation. The KV cache saves computation — but does it save bandwidth? Does it add bandwidth pressure? *Think carefully — this is the setup for Section 2.*

### 🏆 Optional Challenges

If you want to push your understanding further:

**Challenge 1 — Attention Intensity Derivation**
We gave you the formula for attention intensity in the TODO section. Now derive it from scratch for the full attention operation including softmax, the value projection $\text{softmax}(\text{scores}) V$, and the output projection. How does the total intensity of a full attention layer compare to just the $QK^\top$ part?

**Challenge 2 — The KV Cache Bandwidth Cost**
During token generation with a sequence of length $S$, the KV cache stores $2 \times S \times d_k \times n_{\text{heads}}$ elements (K and V for each layer). Compute how many bytes must be loaded from VRAM per token generation step just to read the KV cache. For a 7B parameter model with 32 heads, $d_k = 128$, and $S = 4096$, how does this compare to the bandwidth cost of the model weights themselves?

**Challenge 3 — Flash Attention's Insight**
Standard attention materializes the full $S \times S$ score matrix in VRAM. Flash Attention avoids this by computing attention in tiles. How does this change the arithmetic intensity? Use the formulas from this notebook to quantify the bandwidth savings for $S = 2048$, $d_k = 64$.

**Challenge 4 — Extend the Dashboard**
Modify `create_roofline_dashboard` to show three GPUs side by side in the top row and a combined batch-size analysis in the bottom row. Add a dashed "LLM inference target" line showing where inference "needs to be" for good utilization.

In [ ]:
# A parting thought to carry into Section 2...

print("=" * 60)
print("  Key Takeaway from Section 1")
print("=" * 60)
print()
print("  Token generation is deeply memory-bound.")
print()
print("  Every byte of model weight we load from VRAM")
print("  is used for roughly 1 arithmetic operation.")
print()
print("  The ridge point demands ~156 ops/byte (A100).")
print()
print("  We are at ~0.6% of peak compute utilization.")
print()
print("  The GPU is not the bottleneck. The data bus is.")
print()
print("  In Section 2, we will see how the KV cache changes")
print("  what data needs to travel — and whether that helps.")
print()
print("=" * 60)
print("  → Next: Section 2 — The KV Cache Explained")
print("=" * 60)